# Lyapunov heatmap — trajectories starting on a cat attractor

Same map as `lyapunov_heatmap.ipynb` (sign of the largest Lyapunov exponent over $\varepsilon_p \times \varepsilon_d/g_2$, same parameters,
`t_max = 30/κ_b`, transient `Ttr = 15/κ_b`). The only change is the initial state, which is one of the two cat attractors instead of $a = b = 0$:

$$a(0) = i\,|\alpha|, \qquad b(0) = 0 .$$

**Cat amplitude.** With $\omega_p = 0$ the pump enters as $\sin\varepsilon_p$, and $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$
(12/10/2026; it was $J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$, about half as large). The cat solution of the appendix
($a_{\pm1} = b_2 = b_{-1} = 0$, so that $\beta_b' x^2 = \varepsilon_d'$) then gives, at lowest order in the junction phase,
$$|\alpha|^2 = \frac{\varepsilon_d}{2 g_2}.$$
Before 12/10/2026, with the $J_1$ definition, this read $|\alpha|^2 = (\varepsilon_d/2g_2)\,J_1(\varepsilon_p)/\sin\varepsilon_p$ (`use_exact_cat`), which is the same cat.

The phase $a(0) = \pm i|\alpha|$ is the cat's position at $t = 0$ in the laboratory frame (the cats of `limit_cycles.ipynb`, $\theta \simeq \pm\pi/2$).
`cat_sign` picks one of the two cats; by the $a \to -a$ symmetry both give the same map.

**22/10/2026: start exactly on the cat, up to its fold.** $a(0) = i|\alpha|$, $b(0) = 0$ is not on the cat orbit, and its distance to the cat grows with $g_2$
while the basin of the cat shrinks (`Hill_method_cat_fold.ipynb`). The onset of the map above was 6–9% below the cat fold. To separate the two effects,
the trajectory now starts **on** the cat limit cycle wherever the cat exists: the $\tau = 0$ point of the exact period-1 cat (Hill's method, $N_H = 16$),
computed on the grid of the map by `Hill_cat_seed_grid.ipynb` (`cat_seed_hill_22_10_2026.json`), with
$$a(0) = \frac{x_a + i\,dx_a/d\tau}{\sqrt2\,\varphi_a}, \qquad b(0) = \frac{x_b + i\,(dx_b/d\tau)/2}{\sqrt2\,\varphi_b}.$$
Above the **cat fold** (saddle-node of limit cycles, where the cat disappears with a saddle orbit; $g_2^\ast|\alpha| \approx 219$ MHz) there is no cat,
and the trajectory starts as before, from $a(0) = i|\alpha|$, $b(0) = 0$. For $\varepsilon_d/g_2 \le 6.4$ the fold is above the top of the map, and every point starts on the cat.
The fold line (`cat_fold_line_19_10_2026.json`, from `Hill_method_cat_fold`) is drawn on the map. Question: is the onset of chaos then exactly at the fold, or below it?

In [ ]:
using MPSDynamics, ChaosTools, ProgressBars

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0

In [ ]:
t_max = 30/κ_b
t_w = 15/κ_b                 # transient discarded by lyapunov (Ttr)

cat_sign = 1                 # which cat: a(0) = cat_sign · i|α|

environment = MeanField(κ_a, κ_b)
system_0 = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, 0.0, 0.0, 0.0)
environment_prepared = prepare(system_0, environment)
dt = get_dt(system_0, environment)

cat_amplitude(ϵ_p, ϵ_d_g2_ratio) = sqrt(ϵ_d_g2_ratio/2)   # |α|² = ϵ_d/2g2 with g2 = E_J sin(ϵ_p) φ_a² φ_b

# prepared system and ODE state [ϕ_a, ϕ_b, ϕ̇_a, ϕ̇_b] for a(0) = re_a_0 + i im_a_0, b(0) = 0
function initial_state_cat(ϵ_p, ϵ_d_g2_ratio)
    re_a_0, im_a_0 = 0.0, cat_sign*cat_amplitude(ϵ_p, ϵ_d_g2_ratio)
    ϕ_a_0, q_a_0 = sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
    system_prepared = prepare(SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, 0.0, 0.0))
    u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
    return system_prepared, vcat(u_0, u_dot_0)
end

In [ ]:
ϵ_p_list = vcat(0.2:.01:1.1)
ϵ_d_g2_ratio_list = 2:.1:13.5   # was 5:.1:13.5; extended to overlap the OTOC grid

# Exact cat (Hill) at τ = 0 on this grid, from Hill_cat_seed_grid.ipynb: state_plus[j][i] = [x_a, x_b, dx_a/dτ, dx_b/dτ] (x^λ units)
# for ϵ_d/g2 index j and ϵ_p index i, cat with Im a(0) > 0 (cat_sign = 1); state_minus: the other cat. nothing above the cat fold.
using JSON
cat_seed_data = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/cat_seed_hill_22_10_2026.json")
@assert Float64.(cat_seed_data["eps_p_list"]) ≈ ϵ_p_list && Float64.(cat_seed_data["eps_d_over_g2"]) ≈ collect(ϵ_d_g2_ratio_list)
cat_seed_state = cat_seed_data[cat_sign == 1 ? "state_plus" : "state_minus"]
g2_cat_fold_MHz = [isnothing(g) ? NaN : Float64(g) for g in cat_seed_data["g2_fold_MHz"]]   # NaN: fold above the map

# prepared system and ODE state on the exact cat, or nothing above the fold (same output as initial_state_cat)
function initial_state_hill_cat(j, i)
    st = cat_seed_state[j][i]
    isnothing(st) && return nothing
    x_a, x_b, v_a, v_b = Float64.(st)
    a_0, b_0 = (x_a + 1im*v_a)/(sqrt(2)*φ_a), (x_b + 1im*v_b/2)/(sqrt(2)*φ_b)
    system_prepared = prepare(SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb,
        sqrt(2*Z_a)*real(a_0), sqrt(2/Z_a)*imag(a_0), sqrt(2*Z_b)*real(b_0), sqrt(2/Z_b)*imag(b_0)))
    u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
    return system_prepared, vcat(u_0, u_dot_0)
end

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
on_cat_start = falses(length(ϵ_d_g2_ratio_list), length(ϵ_p_list))   # true: started on the exact cat (below the fold)
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    drive.ϵ_0 = ϵ_p_i
    g2_i = sin(ϵ_p_i)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        seed = initial_state_hill_cat(j, i)                  # on the cat below its fold
        on_cat_start[j,i] = !isnothing(seed)
        system_prepared, u_0_cat = something(seed, initial_state_cat(ϵ_p_i, ϵ_d_g2_ratio_list[j]))   # above the fold: a = i|α|, b = 0
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_cat, p)
        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);
Lyap_coef_chaos_half_sign = map(x -> x>0 ? x : -1, Lyap_coef_chaos);

In [ ]:
using JSON
α_sq = (ϵ_d_g2_ratio_list./2)
g2_list = E_J.*sin.(ϵ_p_list).*φ_a^2 .*φ_b

# chaotic region of the original map (start at a = b = 0), same grid, for comparison
Lyap_origin_sign = hcat(JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/data_lyapunov_12_10_2026.json")...)   # (ϵ_d/g2) × ϵ_p

# chaos onset of the origin-start map: for each ϵ_p (column), smallest ϵ_d/2g2 with a positive exponent
alpha_sq_chaos = [begin
        idx = findfirst(==(1), Lyap_origin_sign[:, i])
        isnothing(idx) ? NaN : α_sq[idx]
    end for i in 1:size(Lyap_origin_sign, 2)]

heatmap(g2_list/2π*1e3, α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds)
plot!(g2_list/2π*1e3, alpha_sq_chaos, color=:yellow, lw=2, label="Chaotic threshold origin")


#contour!(g2_list/2π*1e3, α_sq, Lyap_origin_sign, levels=[0.0], color=:black, lw=1, ls=:dash, colorbar_entry=false, label="")

# instability of the period-2 orbit from Hill_method_doubling_corrected.ipynb (along the ascending ϵ_d/g2 stored next to it)
hill_path = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_12_10_2026_Hill_unstable.json"
if isfile(hill_path)
    plot!(Float64.(JSON.parsefile(hill_path)), Float64.(JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/eps_d_ratio_list_12_10_2026_Hill_unstable.json"))./2, color=:black, lw=2, label="Hill's unstability")
else
    println("No Hill line: run the 2D sweep and the save cell of Hill_method_doubling_corrected.ipynb to create ", hill_path)
end

# fold of the cat orbit from Hill_method_cat_fold.ipynb (NaN: no fold below g2_max)
cat_fold = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/cat_fold_line_19_10_2026.json")
plot!([isnothing(g) ? NaN : Float64(g) for g in cat_fold["g2_fold_MHz"]], Float64.(cat_fold["eps_d_over_g2"])./2,
      color=:black, ls=:dash, lw=2, label="Cat fold (Hill)")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\varepsilon_d/2g_2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2π*1e3),maximum(g2_list/2π*1e3)),ylims=(minimum(α_sq),maximum(α_sq)+2),
    colorbar_title="\n\nSign of Lyapunov exp.",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/heatmap_lyapunov_cat_start_22_10_2026.pdf")
current()

### Onset of chaos against the cat fold
For each $\varepsilon_d/g_2$: the first $g_2$ of the grid with $\lambda > 0$, against the cat fold of the same column (from the seed grid, exact on this $\varepsilon_d/g_2$).
The grid step in $g_2$ is $\approx 1$ MHz at the fold ($\Delta\varepsilon_p = 0.01$). "Onset − fold" in units of grid points: 1 means chaos at the first grid point above the fold.

In [ ]:
g2_MHz_grid = g2_list/2π*1e3
first_chaotic(j) = (k = findfirst(>(0), Lyap_coef_chaos[j, :]); isnothing(k) ? NaN : g2_MHz_grid[k])
g2_onset_MHz = [first_chaotic(j) for j in eachindex(ϵ_d_g2_ratio_list)]
# grid points between the fold and the onset: number of ϵ_p of the grid in (fold, onset]
steps_above_fold = [isnan(g2_cat_fold_MHz[j]) || isnan(g2_onset_MHz[j]) ? NaN :
                    count(g -> g2_cat_fold_MHz[j] < g <= g2_onset_MHz[j], g2_MHz_grid) * sign(g2_onset_MHz[j] - g2_cat_fold_MHz[j])
                    for j in eachindex(ϵ_d_g2_ratio_list)]
println("ϵ_d/g2   cat fold   onset (MHz)   onset - fold (grid points)   chaotic points below the fold")
for j in 1:5:length(ϵ_d_g2_ratio_list)
    n_below = count(i -> on_cat_start[j, i] && Lyap_coef_chaos[j, i] > 0, eachindex(ϵ_p_list))
    println(rpad(ϵ_d_g2_ratio_list[j], 9), rpad(round(g2_cat_fold_MHz[j], digits=2), 11), rpad(round(g2_onset_MHz[j], digits=1), 14),
            rpad(steps_above_fold[j], 29), n_below)
end
ok = .!isnan.(steps_above_fold)
println("\nratios with a fold inside the map: ", count(ok), "; onset - fold in grid points: ",
        Dict(k => count(==(k), steps_above_fold[ok]) for k in unique(steps_above_fold[ok])))
println("chaotic points started on the cat (below the fold): ", count(on_cat_start .& (Lyap_coef_chaos .> 0)), " / ", count(on_cat_start))

Same grid and file format as `data_lyapunov_12_10_2026.json` (outer index: ϵ_p), so `figure_lyapunov_heatmap.ipynb` can read it. Since 22/10/2026 the files carry the suffix `_22_10_2026` (start on the exact cat below its fold); the `_12_10_2026` files are the map from $a(0) = i|\alpha|$ everywhere.

In [ ]:
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/data_lyapunov_cat_start_22_10_2026.json", "w") do io
    JSON.print(io, Lyap_coef_chaos_sign)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/g2_list_cat_start_22_10_2026.json", "w") do io
    JSON.print(io, g2_list)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/alpha_sq_list_cat_start_22_10_2026.json", "w") do io
    JSON.print(io, α_sq)
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/lyapunov_values_cat_start_22_10_2026.json", "w") do io
    JSON.print(io, Lyap_coef_chaos)      # λ (1/ns), same layout
end
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/on_cat_start_22_10_2026.json", "w") do io
    JSON.print(io, on_cat_start)         # true: started on the exact cat (below its fold), false: a = i|α|, b = 0
end